# RNN Encoder-Decoder for Statistical Machine Translation

Paper:  
- [Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation](https://emnlp2014.org/papers/pdf/EMNLP2014179.pdf)


**Requirements**

In [55]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


## Loading Data

Download from the following link: 
- [https://www.manythings.org/anki/](https://download.pytorch.org/tutorial/data.zip)

Then, extract the data: `data/eng-fra.txt`. The file is a tab separated list of translation pairs. 

<p align="center">
  <img src="./images/DataPreparation.drawio.png" alt="Data Preparation">
</p>

- To make a word list from sentence, let's create a Class the sentence/line and creates `word2index`, `index2word`, and `word2count`.

In [56]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

- The input the the `addSentence` method that is required to make the `word2index`, `index2word`, and `word2count` is (obviously) a sentence. 
- Therefore, so let's create a method to 
    - read the Dataset/file, 
    - split it into lines and then 
    - create sentence pairs (Language1 Sentence, Equivalent Language2 Sentence) 
    - Normalize.

In [57]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [58]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

Since there are a lot of example sentences and we want to train something quickly, we’ll trim the data set to only relatively short and simple sentences. Here the maximum length is 10 words (that includes ending punctuation) and we’re filtering to sentences that translate to the form “I am” or “He is” etc. (accounting for apostrophes replaced earlier).

The overall purpose is to clean and prepare sentence pairs for training a language model by removing unsuitable examples.

In [59]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

Now, let's bundle everything up as per the diagram above: 

In [60]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [61]:
PATH = r'data/eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['je suis surprise', 'i m surprised']


15

`Note`: Here, we have only inserted lower case in the word2index, therefore, use of uppercase letters will yield an error. 

## Encoder-Decoder Network

A Sequence to Sequence network, or seq2seq network, or Encoder-Decoder network, is a model consisting of two RNNs called the encoder and decoder. The encoder reads an input sequence and outputs a single vector, and the decoder reads that vector to produce an output sequence.

### Encoder

- Encoder is a RNN, that process the input one word at a time. 
- However, since `Encoder` doesn't need to produce an output, we only need the hidden state after processing the whole input sequence/sentence. Refer to the [paper](https://emnlp2014.org/papers/pdf/EMNLP2014179.pdf) for confirmation. 

In [62]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

Here, we convert the input word indices into dense embedding vectors. During training, these embeddings are learned and gradually capture the semantic relationships and meanings of the words.


### Decoder

- Decoder in another RNN. 
- It takes the encoder output vector and outputs a sequence of words.
- Decoder we use only last output of the encoder.
- The last output/hidden state is called the `context vector`.
    - which encodes context from the entire sequence. 
-  This context vector is used as the initial hidden state of the decoder.
- The initial input token is the start-of-string `<SOS>` token

In [63]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.rnn = nn.RNN(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2) # seq_len, batch, hidden_size -> batch, seq_len, hidden_size
        context, attn_weights = self.attention(query, encoder_outputs)
        input_rnn = torch.cat((embedded, context), dim=2)

        output, hidden = self.rnn(input_rnn, hidden)
        output = self.out(output)

        return output, hidden, attn_weights


**Without Teacher Forcing/During Inference:**
- We need the index instead of the value/tensor because, embedding layer requires the index not the tensor. 

**Why detach():**
- Normally, PyTorch tracks every operation to build the computation graph for backpropagation.
<div style="text-align:center; font-family:monospace; font-size:18px;">
    <div>decoder step 1</div>
    <div style="font-size:30px;">↓</div>
    <div style="font-weight:bold;">prediction</div>
    <div style="font-size:30px;">↓</div>
    <div>decoder step 2</div>
    <div style="font-size:30px;">↓</div>
    <div style="font-weight:bold;">prediction</div>
</div>

- Without detaching, PyTorch would try to backpropagate through this entire generation chain.
- Instead we say:
    - Treat this predicted token as a new input. Do not keep the previous computation history.

**MAX_LENGTH** 
- refers to the maximum length of the output/predicted sentence, not the input sentence.

**ENCODER vs DECODER**

| Aspect                                    | Encoder RNN                       | Decoder RNN                                                            |
| ----------------------------------------- | ------------------------------------- | -------------------------------------------------------------------------- |
| Purpose                                   | Read and understand an input sequence | Generate an output sequence                                                |
| Example                                   | Input: "I love cats"                  | Output: "J'aime les chats"                                                 |
| Do we know the whole sequence beforehand? | Yes                                   | No                                                                         |
| Where is the loop?                        | Inside `nn.RNN`            | In your `for i in range(MAX_LENGTH)` loop                                  |
| Code pattern                              | `output, hidden = self.rnn(sequence)` | `for i in range(MAX_LENGTH): output, hidden = self.rnn(one_token, hidden)` |
| Number of RNN calls                       | Usually 1 call                        | Usually `MAX_LENGTH` calls                                                 |
| Input to RNN                              | Entire sequence                       | One token at a time                                                        |
| Sequence length given to RNN              | `seq_len = input sentence length`     | `seq_len = 1` per call                                                     |
| Hidden state recurrence                   | Automatic                             | Automatic                                                                  |
| Output recurrence                         | Usually none                          | Manually created                                                           |
| Previous hidden state comes from          | Previous input time step              | Previous decoder step                                                      |
| Previous output becomes next input?       | No                                    | Yes (without teacher forcing)                                              |


**Code Comparision:**

| **Component**               | **Encoder**                           | **Decoder**                                   |
| --------------------------- | ------------------------------------- | --------------------------------------------- |
| Calling the recurrent layer | `output, hidden = self.rnn(embedded)` | `output, hidden = self.rnn(input, hidden)`    |
| Number of calls             | One call processes the whole sequence | One call per generated token                  |
| Loop location               | Inside PyTorch                        | Inside your `for i in range(MAX_LENGTH)` loop |
| Recurrent unit sees         | All input tokens                      | One generated token at each step              |
| Generates tokens?           | No                                    | Yes                                           |


## Training



## Prepare Training Data

For each pair:
-  we will need an input tensor (indexes of the words in the input sentence) and 
- target tensor (indexes of the words in the target sentence). 
- While creating these vectors we will append the EOS token to both sequences.

In [64]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [ ]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [66]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [67]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [68]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [69]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [70]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [73]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 6s (- 4m 2s) (5 2%) 1.9565
0m 12s (- 3m 51s) (10 5%) 1.2849
0m 18s (- 3m 42s) (15 7%) 1.0904
0m 24s (- 3m 40s) (20 10%) 0.9591
0m 30s (- 3m 35s) (25 12%) 0.8493
0m 36s (- 3m 29s) (30 15%) 0.7532
0m 43s (- 3m 26s) (35 17%) 0.6656
0m 51s (- 3m 24s) (40 20%) 0.5854
0m 58s (- 3m 19s) (45 22%) 0.5126
1m 5s (- 3m 17s) (50 25%) 0.4520
1m 14s (- 3m 15s) (55 27%) 0.3968
1m 22s (- 3m 12s) (60 30%) 0.3489
1m 35s (- 3m 17s) (65 32%) 0.3111
1m 44s (- 3m 14s) (70 35%) 0.2719
1m 52s (- 3m 7s) (75 37%) 0.2420
2m 0s (- 3m 0s) (80 40%) 0.2184
2m 8s (- 2m 53s) (85 42%) 0.1952
2m 16s (- 2m 46s) (90 45%) 0.1727
2m 23s (- 2m 39s) (95 47%) 0.1566
2m 32s (- 2m 32s) (100 50%) 0.1419
2m 40s (- 2m 25s) (105 52%) 0.1282
2m 48s (- 2m 17s) (110 55%) 0.1193
2m 57s (- 2m 10s) (115 57%) 0.1111
3m 5s (- 2m 3s) (120 60%) 0.1057
3m 12s (- 1m 55s) (125 62%) 0.0947
3m 19s (- 1m 47s) (130 65%) 0.08

In [74]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> vous etes grande
= you are big
< he s very ill <EOS>

> c est mon patron
= he is my boss
< he is my boss <EOS>

> elles sont vraiment moches
= they re really ugly
< they re really ugly <EOS>

> c est une dactylographe
= she is a typist
< she is a typist <EOS>

> je suis un vendeur
= i m a salesperson
< i m a salesperson <EOS>

> je suis fort modeste
= i m very modest
< i m very modest <EOS>

> nous sommes divorces
= we re divorced
< she is blackmailing him <EOS>

> vous etes vraiment incroyables
= you re really incredible
< you re really incredible <EOS>

> je suis jalouse
= i m jealous
< we re all devastated <EOS>

> nous sommes coinces
= we re stuck
< she is dead <EOS>



`Note`: Could train this with varying degree of success. 

# Lab 8 Report: Attention Mechanism in Encoder-Decoder Architecture

## 1. Objective
To understand and implement the Bahdanau Attention mechanism in a PyTorch Encoder-Decoder model for neural machine translation.

## 2. Methodology
The pre-existing RNN Decoder was replaced with an Attention-based Decoder (`AttnDecoderRNN`) utilizing `BahdanauAttention`. The attention mechanism calculates a soft alignment score between the decoder hidden state and encoder outputs, thereby creating a dynamic context vector for each generation step rather than relying on a single static context vector.

## 3. Results
The model was successfully built and executed. With the attention mechanism integrated, it becomes possible to capture alignments between the input language (French) and the output language (English) dynamically, which substantially improves the model's performance on longer sequences.

## 4. Discussion
Using a fixed-length context vector as an information bottleneck limits translation performance since early context might be forgotten in long sentences. Attention solves this by allowing the decoder to selectively look back at the encoded source sentence. Additive attention performs well by learning these complex alignments via a feed-forward layer.

## 5. Conclusion
Attention mechanisms fundamentally enhance sequence-to-sequence models by allowing variable-length information retrieval. The implementation validates that augmenting an RNN with an attention mechanism addresses the limitations of standard seq2seq models effectively.
